# 01 — Bronze Layer: Raw Data Ingestion

This notebook demonstrates the **Bronze layer** of the B3 Data Platform.

**Goal:** ingest raw daily OHLCV prices from Yahoo Finance and persist them as-is
(no transformation) into the Bronze Parquet store.

Architecture reminder:
```
Yahoo Finance API  ──►  Bronze (raw Parquet)  ──►  Silver  ──►  Gold
```
**Rules for Bronze:**
- Data is written exactly as received — no value changes.
- Only metadata columns are added: `source` and `ingested_at`.
- Partitioned by `trade_date` for fast downstream reads.

In [1]:
import sys
sys.path.insert(0, "..")

from datetime import date, timedelta

import polars as pl

pl.Config.set_tbl_rows(20)
print(f"Polars version: {pl.__version__}")

Polars version: 1.42.0


## 1. Fetch raw prices from Yahoo Finance

In [5]:
from c_ingestion.yahoo_finance import fetch_daily_prices

END   = date.today()
START = END - timedelta(days=90)  # last 3 months

# Use a small subset for the demo
DEMO_TICKERS = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "ABEV3.SA"]

raw_df = fetch_daily_prices(tickers=DEMO_TICKERS, start=START, end=END)
print(f"Fetched {len(raw_df):,} rows for {raw_df['ticker'].n_unique()} tickers")
raw_df.head(10)

{"timestamp": "2026-06-25T16:01:25.097346+00:00", "level": "INFO", "logger": "c_ingestion.yahoo_finance", "message": "Starting Yahoo Finance ingestion", "module": "yahoo_finance", "func": "fetch_daily_prices", "line": 96, "name": "c_ingestion.yahoo_finance", "msg": "Starting Yahoo Finance ingestion", "args": [], "levelname": "INFO", "levelno": 20, "pathname": "\\\\wsl.localhost\\ubuntu-llm\\home\\efcardoso\\projects\\b3-data-plataform\\i_notebooks\\..\\c_ingestion\\yahoo_finance.py", "filename": "yahoo_finance.py", "exc_info": null, "exc_text": null, "stack_info": null, "lineno": 96, "funcName": "fetch_daily_prices", "created": 1782403285.0972726, "msecs": 97.0, "relativeCreated": 1383309.8572, "thread": 22476, "threadName": "MainThread", "processName": "MainProcess", "process": 23756, "taskName": "Task-122", "tickers": 5, "start": "2026-03-27", "end": "2026-06-25"}
{"timestamp": "2026-06-25T16:01:25.392926+00:00", "level": "INFO", "logger": "c_ingestion.yahoo_finance", "message": "Ing

ticker,trade_date,open,high,low,close,adj_close,volume
str,date,f64,f64,f64,f64,f64,i64
"""PETR4.SA""",2026-03-27,48.5,49.439999,48.130001,49.41,47.916618,54170900
"""PETR4.SA""",2026-03-30,49.75,50.689999,49.490002,49.669998,48.168758,51454300
"""PETR4.SA""",2026-03-31,50.07,50.549999,47.650002,48.669998,47.198982,78931700
"""PETR4.SA""",2026-04-01,47.700001,48.07,46.77,47.369999,45.938274,66488400
"""PETR4.SA""",2026-04-02,49.299999,49.459999,47.950001,48.150002,46.694702,38536900
"""PETR4.SA""",2026-04-06,47.990002,49.0,47.880001,48.939999,47.460823,27848100
"""PETR4.SA""",2026-04-07,49.09,49.560001,48.400002,48.509998,47.043819,33362800
"""PETR4.SA""",2026-04-08,44.700001,46.639999,44.529999,46.610001,45.201248,87580500
"""PETR4.SA""",2026-04-09,47.540001,48.549999,47.029999,47.900002,46.452259,58206500


## 2. Inspect raw data

In [7]:
print("Schema:")
print(raw_df.schema)
print("\nNull counts:")
print(raw_df.null_count())
print("\nBasic stats:")
raw_df.describe()

Schema:
Schema({'ticker': String, 'trade_date': Date, 'open': Float64, 'high': Float64, 'low': Float64, 'close': Float64, 'adj_close': Float64, 'volume': Int64})

Null counts:
shape: (1, 8)
┌────────┬────────────┬──────┬──────┬─────┬───────┬───────────┬────────┐
│ ticker ┆ trade_date ┆ open ┆ high ┆ low ┆ close ┆ adj_close ┆ volume │
│ ---    ┆ ---        ┆ ---  ┆ ---  ┆ --- ┆ ---   ┆ ---       ┆ ---    │
│ u32    ┆ u32        ┆ u32  ┆ u32  ┆ u32 ┆ u32   ┆ u32       ┆ u32    │
╞════════╪════════════╪══════╪══════╪═════╪═══════╪═══════════╪════════╡
│ 0      ┆ 0          ┆ 0    ┆ 0    ┆ 0   ┆ 0     ┆ 0         ┆ 0      │
└────────┴────────────┴──────┴──────┴─────┴───────┴───────────┴────────┘

Basic stats:


statistic,ticker,trade_date,open,high,low,close,adj_close,volume
str,str,str,f64,f64,f64,f64,f64,f64
"""count""","""300""","""300""",300.0,300.0,300.0,300.0,300.0,300.0
"""null_count""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,"""2026-05-11 15:12:00""",40.785267,41.224933,40.3246,40.7627,40.518635,3.1334303e7
"""std""",null,null,24.138005,24.394588,23.854262,24.149731,24.12708,1.6411e7
"""min""","""ABEV3.SA""","""2026-03-27""",14.36,14.56,14.32,14.33,14.290209,7.0595e6
"""25%""",null,"""2026-04-20""",17.620001,17.93,17.5,17.66,17.66,2.01966e7
"""50%""",null,"""2026-05-13""",40.290001,40.650002,40.099998,40.400002,40.020744,2.67209e7
"""75%""",null,"""2026-06-02""",47.639999,48.169998,47.169998,47.77,46.586304,3.70226e7
"""max""","""VALE3.SA""","""2026-06-24""",88.889999,89.75,88.029999,89.75,89.75,1.099523e8


In [8]:
# Date coverage per ticker
raw_df.group_by("ticker").agg(
    pl.col("trade_date").min().alias("first_date"),
    pl.col("trade_date").max().alias("last_date"),
    pl.col("trade_date").n_unique().alias("trading_days"),
).sort("ticker")

ticker,first_date,last_date,trading_days
str,date,date,u32
"""ABEV3.SA""",2026-03-27,2026-06-24,60
"""BBDC4.SA""",2026-03-27,2026-06-24,60
"""ITUB4.SA""",2026-03-27,2026-06-24,60
"""PETR4.SA""",2026-03-27,2026-06-24,60
"""VALE3.SA""",2026-03-27,2026-06-24,60


## 3. Write to Bronze layer

In [9]:
from d_processing.bronze.ingest import write_bronze

bronze_path = write_bronze(raw_df, source="yahoo_finance")
print(f"Written to: {bronze_path}")

{"timestamp": "2026-06-25T16:01:46.017867+00:00", "level": "INFO", "logger": "d_processing.bronze.ingest", "message": "Bronze write complete", "module": "ingest", "func": "write_bronze", "line": 62, "name": "d_processing.bronze.ingest", "msg": "Bronze write complete", "args": [], "levelname": "INFO", "levelno": 20, "pathname": "\\\\wsl.localhost\\ubuntu-llm\\home\\efcardoso\\projects\\b3-data-plataform\\i_notebooks\\..\\d_processing\\bronze\\ingest.py", "filename": "ingest.py", "exc_info": null, "exc_text": null, "stack_info": null, "lineno": 62, "funcName": "write_bronze", "created": 1782403306.0177903, "msecs": 17.0, "relativeCreated": 1404230.3748, "thread": 22476, "threadName": "MainThread", "processName": "MainProcess", "process": 23756, "taskName": "Task-173", "source": "yahoo_finance", "rows": 300, "partitions": 60, "output": "\\\\wsl.localhost\\ubuntu-llm\\home\\efcardoso\\projects\\b3-data-plataform\\j_data\\bronze\\yahoo_finance", "ingested_at": "2026-06-25T16:01:42.536142+00

## 4. Read back from Bronze and verify

In [10]:
from d_processing.bronze.ingest import read_bronze

bronze_df = read_bronze(source="yahoo_finance")
print(f"Bronze rows: {len(bronze_df):,}")
print(f"Columns added: {[c for c in bronze_df.columns if c not in raw_df.columns]}")
bronze_df.head(10)

{"timestamp": "2026-06-25T16:01:52.386942+00:00", "level": "INFO", "logger": "d_processing.bronze.ingest", "message": "Bronze read complete", "module": "ingest", "func": "read_bronze", "line": 107, "name": "d_processing.bronze.ingest", "msg": "Bronze read complete", "args": [], "levelname": "INFO", "levelno": 20, "pathname": "\\\\wsl.localhost\\ubuntu-llm\\home\\efcardoso\\projects\\b3-data-plataform\\i_notebooks\\..\\d_processing\\bronze\\ingest.py", "filename": "ingest.py", "exc_info": null, "exc_text": null, "stack_info": null, "lineno": 107, "funcName": "read_bronze", "created": 1782403312.3868873, "msecs": 386.0, "relativeCreated": 1410599.472, "thread": 22476, "threadName": "MainThread", "processName": "MainProcess", "process": 23756, "taskName": "Task-188", "rows": 300, "source": "yahoo_finance"}
Bronze rows: 300
Columns added: ['source', 'ingested_at']


ticker,trade_date,open,high,low,close,adj_close,volume,source,ingested_at
str,date,f64,f64,f64,f64,f64,i64,str,"datetime[μs, UTC]"
"""PETR4.SA""",2026-03-27,48.5,49.439999,48.130001,49.41,47.916618,54170900,"""yahoo_finance""",2026-06-25 16:01:42.536142 UTC
"""VALE3.SA""",2026-03-27,78.470001,79.919998,78.169998,79.0,79.0,11101700,"""yahoo_finance""",2026-06-25 16:01:42.536142 UTC
"""ITUB4.SA""",2026-03-27,41.810001,41.810001,41.18,41.450001,41.026432,27910700,"""yahoo_finance""",2026-06-25 16:01:42.536142 UTC
"""BBDC4.SA""",2026-03-27,18.709999,18.809999,18.440001,18.52,18.1793,20727200,"""yahoo_finance""",2026-06-25 16:01:42.536142 UTC
"""ABEV3.SA""",2026-03-27,14.79,15.0,14.74,14.76,14.719015,17884700,"""yahoo_finance""",2026-06-25 16:01:42.536142 UTC
"""PETR4.SA""",2026-03-30,49.75,50.689999,49.490002,49.669998,48.168758,51454300,"""yahoo_finance""",2026-06-25 16:01:42.536142 UTC
"""VALE3.SA""",2026-03-30,80.25,81.059998,79.290001,79.5,79.5,18776700,"""yahoo_finance""",2026-06-25 16:01:42.536142 UTC
"""ITUB4.SA""",2026-03-30,42.0,42.060001,41.23,41.599998,41.174904,18528100,"""yahoo_finance""",2026-06-25 16:01:42.536142 UTC
"""BBDC4.SA""",2026-03-30,18.77,18.790001,18.379999,18.469999,18.13022,32459400,"""yahoo_finance""",2026-06-25 16:01:42.536142 UTC


## 5. Quick visualization — closing prices

In [ ]:
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_white"

fig = px.line(
    bronze_df.sort("trade_date").to_pandas(),
    x="trade_date",
    y="close",
    color="ticker",
    title="Preços de Fechamento das Ações da B3 — Camada Bronze (dados brutos)",
    labels={
        "trade_date": "Data do Pregão",
        "close": "Preço de Fechamento (R$)",
        "ticker": "Ativo",
    },
)
fig.update_layout(
    height=500,
    hovermode="x unified",
    legend=dict(title="Ativo", orientation="v", x=1.02, y=1),
    margin=dict(l=60, r=40, t=70, b=50),
)
fig.update_yaxes(tickprefix="R$ ", separatethousands=True)
fig.update_xaxes(tickformat="%d/%m/%Y")
fig.show()

## 6. Run full Bronze pipeline (all default tickers)

In [12]:
from f_pipelines.bronze_pipeline import BronzePipeline, BronzePipelineConfig

cfg = BronzePipelineConfig(start=START, end=END)
pipeline = BronzePipeline(config=cfg)
result = pipeline.run()
print(f"Pipeline produced {len(result):,} rows")

{"timestamp": "2026-06-25T16:02:54.509172+00:00", "level": "INFO", "logger": "f_pipelines.bronze_pipeline", "message": "BronzePipeline started", "module": "bronze_pipeline", "func": "run", "line": 59, "name": "f_pipelines.bronze_pipeline", "msg": "BronzePipeline started", "args": [], "levelname": "INFO", "levelno": 20, "pathname": "\\\\wsl.localhost\\ubuntu-llm\\home\\efcardoso\\projects\\b3-data-plataform\\i_notebooks\\..\\f_pipelines\\bronze_pipeline.py", "filename": "bronze_pipeline.py", "exc_info": null, "exc_text": null, "stack_info": null, "lineno": 59, "funcName": "run", "created": 1782403374.5091324, "msecs": 509.0, "relativeCreated": 1472721.7169, "thread": 22476, "threadName": "MainThread", "processName": "MainProcess", "process": 23756, "taskName": "Task-224"}
{"timestamp": "2026-06-25T16:02:54.511303+00:00", "level": "INFO", "logger": "f_pipelines.bronze_pipeline", "message": "Bronze extraction started", "module": "bronze_pipeline", "func": "extract", "line": 41, "name": "f